In [13]:
# ===============================
# 📌 STEP 0: Imports
# ===============================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import xgboost as xgb
import os

# ===============================
# 📌 STEP 1: Load embeddings
# ===============================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ===============================
# 📌 STEP 2: Align train/test sizes
# ===============================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ===============================
# 📌 STEP 3: Combine embeddings
# ===============================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ===============================
# 📌 STEP 4: Define SMAPE metric
# ===============================
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.mean(diff) * 100

# ===============================
# 📌 STEP 5: K-Fold + Chunked GPU Training using xgb.train
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000

oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    fold_models = []
    
    for i in range(0, len(X_tr), chunk_size):
        print(f"  Training chunk {i//chunk_size + 1}")
        X_chunk = X_tr[i:i+chunk_size]
        y_chunk = y_tr[i:i+chunk_size]
        
        # Convert to DMatrix (required for xgb.train)
        dtrain = xgb.DMatrix(X_chunk, label=y_chunk)
        dval   = xgb.DMatrix(X_val, label=y_val)
        
        params = {
            'objective': 'reg:squarederror',
            'learning_rate': 0.05,
            'max_depth': 8,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'tree_method': 'gpu_hist',  # ✅ GPU
            'eval_metric': 'mae',
            'seed': 42
        }
        
        evals = [(dtrain, 'train'), (dval, 'eval')]
        
        bst = xgb.train(
            params,
            dtrain,
            num_boost_round=2000,
            evals=evals,
            early_stopping_rounds=100,
            verbose_eval=100
        )
        
        fold_models.append(bst)
    
    # OOF prediction for this fold
    fold_oof = np.mean([m.predict(dval) for m in fold_models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction for this fold
    dtest = xgb.DMatrix(test_features)
    fold_test = np.mean([m.predict(dtest) for m in fold_models], axis=0)
    test_preds += fold_test / kf.n_splits

# ===============================
# 📌 STEP 6: Evaluate OOF SMAPE
# ===============================
print("\n✅ OOF SMAPE:", smape(y, oof_preds))

# ===============================
# 📌 STEP 7: Save submission
# ===============================
submission_file = "submission_xgb_gpu_train.csv"
submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved as {submission_file}")



🌟 Fold 1
  Training chunk 1


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [18:28:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()


[0]	train-mae:17.82754	eval-mae:18.59287
[100]	train-mae:9.50915	eval-mae:17.43416
[200]	train-mae:5.75739	eval-mae:17.25448
[300]	train-mae:3.64193	eval-mae:17.18634
[400]	train-mae:2.35996	eval-mae:17.15530
[500]	train-mae:1.52763	eval-mae:17.13521
[600]	train-mae:1.02825	eval-mae:17.12253
[700]	train-mae:0.67172	eval-mae:17.11845
[800]	train-mae:0.46455	eval-mae:17.11422
[900]	train-mae:0.33151	eval-mae:17.11191
[1000]	train-mae:0.23462	eval-mae:17.11087
[1100]	train-mae:0.16726	eval-mae:17.11027
[1200]	train-mae:0.12347	eval-mae:17.10982
[1300]	train-mae:0.09148	eval-mae:17.10930
[1400]	train-mae:0.06759	eval-mae:17.10893
[1500]	train-mae:0.05210	eval-mae:17.10874
[1600]	train-mae:0.03967	eval-mae:17.10860
[1700]	train-mae:0.03043	eval-mae:17.10831
[1800]	train-mae:0.02419	eval-mae:17.10828
[1900]	train-mae:0.01995	eval-mae:17.10819
[1925]	train-mae:0.01882	eval-mae:17.10821
  Training chunk 2


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [18:34:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()


[0]	train-mae:18.22302	eval-mae:18.67314
[100]	train-mae:10.71876	eval-mae:17.46064
[200]	train-mae:6.66217	eval-mae:17.24032
[300]	train-mae:4.14083	eval-mae:17.17563
[400]	train-mae:2.70538	eval-mae:17.15165
[500]	train-mae:1.77502	eval-mae:17.13765
[600]	train-mae:1.17099	eval-mae:17.12604
[700]	train-mae:0.78507	eval-mae:17.11809
[800]	train-mae:0.52422	eval-mae:17.11515
[900]	train-mae:0.35546	eval-mae:17.11280
[1000]	train-mae:0.24166	eval-mae:17.11088
[1100]	train-mae:0.16969	eval-mae:17.11071
[1200]	train-mae:0.12101	eval-mae:17.10950
[1300]	train-mae:0.08827	eval-mae:17.10904
[1400]	train-mae:0.06427	eval-mae:17.10880
[1500]	train-mae:0.04775	eval-mae:17.10861
[1600]	train-mae:0.03513	eval-mae:17.10851
[1700]	train-mae:0.02578	eval-mae:17.10845
[1800]	train-mae:0.01915	eval-mae:17.10844
[1842]	train-mae:0.01696	eval-mae:17.10845
  Training chunk 3


...

[0]	train-mae:18.40855	eval-mae:18.57945
[100]	train-mae:10.86292	eval-mae:16.79984
[200]	train-mae:6.48109	eval-mae:16.59020
[300]	train-mae:4.02819	eval-mae:16.53886
[400]	train-mae:2.58697	eval-mae:16.52068
[500]	train-mae:1.65815	eval-mae:16.50900
[600]	train-mae:1.09389	eval-mae:16.49945
[700]	train-mae:0.70290	eval-mae:16.49212
[800]	train-mae:0.46017	eval-mae:16.48965
[900]	train-mae:0.29246	eval-mae:16.48804
[1000]	train-mae:0.19271	eval-mae:16.48644
[1100]	train-mae:0.12616	eval-mae:16.48608
[1200]	train-mae:0.08297	eval-mae:16.48564
[1300]	train-mae:0.05467	eval-mae:16.48563
[1331]	train-mae:0.04829	eval-mae:16.48551
  Training chunk 4


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [19:59:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()


[0]	train-mae:18.43855	eval-mae:18.60402
[100]	train-mae:10.96683	eval-mae:16.97904
[200]	train-mae:6.76148	eval-mae:16.80827
[300]	train-mae:4.45369	eval-mae:16.75888
[400]	train-mae:2.86072	eval-mae:16.73765
[500]	train-mae:1.81105	eval-mae:16.72222
[600]	train-mae:1.16580	eval-mae:16.71354
[700]	train-mae:0.74053	eval-mae:16.70952
[800]	train-mae:0.48700	eval-mae:16.70546
[900]	train-mae:0.31055	eval-mae:16.70213
[1000]	train-mae:0.20148	eval-mae:16.70093
[1100]	train-mae:0.13080	eval-mae:16.70000
[1200]	train-mae:0.08740	eval-mae:16.69964
[1300]	train-mae:0.05618	eval-mae:16.69934
[1400]	train-mae:0.03636	eval-mae:16.69920
[1500]	train-mae:0.02353	eval-mae:16.69910
[1600]	train-mae:0.01515	eval-mae:16.69909
[1700]	train-mae:0.00974	eval-mae:16.69900
[1800]	train-mae:0.00645	eval-mae:16.69899
[1849]	train-mae:0.00526	eval-mae:16.69899


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\core.py:729: UserWarning: [20:05:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\core.py:729: UserWarning: [20:05:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)



✅ OOF SMAPE: 70.00880383799296
✅ Submission saved as submission_xgb_gpu_train.csv


In [9]:
import xgboost as xgb
print(xgb.__version__)

3.0.5


In [5]:
!pip install xgboost --upgrade

In [ ]:
# ==============================================================
# 🏆 Final Optimized XGBoost (GPU) Training for Lowest SMAPE
# ==============================================================

# STEP 0: Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# ==============================================================
# STEP 1: Load embeddings
# ==============================================================
train_text_embs = np.load("train_text_embeddings_fp16_v2.npy").astype(np.float32)
test_text_embs  = np.load("test_text_embeddings_fp16_v2.npy").astype(np.float32)

train_img_df = pd.read_csv("train_features.csv")
test_img_df  = pd.read_csv("test_features.csv")

img_cols = [c for c in train_img_df.columns if c != 'sample_id']
train_img_features = train_img_df[img_cols].astype(np.float32).values
test_img_features  = test_img_df[img_cols].astype(np.float32).values

# ==============================================================
# STEP 2: Align train/test sizes
# ==============================================================
min_train = min(train_text_embs.shape[0], train_img_features.shape[0])
train_text_embs = train_text_embs[:min_train]
train_img_features = train_img_features[:min_train]

min_test = min(test_text_embs.shape[0], test_img_features.shape[0])
test_text_embs = test_text_embs[:min_test]
test_img_features = test_img_features[:min_test]

# ==============================================================
# STEP 3: Combine embeddings + scaling
# ==============================================================
train_features = np.hstack([train_text_embs, train_img_features])
test_features  = np.hstack([test_text_embs, test_img_features])

scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

train_csv = pd.read_csv("dataset/train.csv")
y = train_csv['price'].values[:min_train]

# ==============================================================
# STEP 4: Define SMAPE metric
# ==============================================================
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.mean(diff) * 100

# ==============================================================
# STEP 5: Chunked K-Fold GPU Training
# ==============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
chunk_size = 15000

oof_preds = np.zeros(len(train_features))
test_preds = np.zeros(len(test_features))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_features)):
    print(f"\n🌟 Fold {fold+1}")
    
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    fold_models = []
    
    for start in range(0, len(X_tr), chunk_size):
        end = start + chunk_size
        X_chunk = X_tr[start:end]
        y_chunk = y_tr[start:end]
        
        dtrain = xgb.DMatrix(X_chunk, label=y_chunk)
        dval   = xgb.DMatrix(X_val, label=y_val)
        
        params = {
            'objective': 'reg:squarederror',
            'learning_rate': 0.03,        # slightly higher for faster learning
            'max_depth': 8,               # deeper trees capture more interaction
            'subsample': 0.85,            # slightly higher to reduce bias
            'colsample_bytree': 0.8,
            'min_child_weight': 2,
            'lambda': 1.2,
            'alpha': 0.5,
            'tree_method': 'gpu_hist',    # GPU acceleration
            'predictor': 'gpu_predictor',
            'seed': 42
        }
        
        evals = [(dtrain, 'train'), (dval, 'eval')]
        
        bst = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=3000,
            evals=evals,
            early_stopping_rounds=150,
            verbose_eval=100
        )
        
        # manual SMAPE evaluation
        val_preds = bst.predict(dval)
        fold_smape = smape(y_val, val_preds)
        print(f"✅ Fold {fold+1} SMAPE: {fold_smape:.4f}")
        
        fold_models.append(bst)
    
    # OOF prediction for validation
    dval = xgb.DMatrix(X_val)
    fold_oof = np.mean([m.predict(dval) for m in fold_models], axis=0)
    oof_preds[val_idx] = fold_oof
    
    # Test prediction
    dtest = xgb.DMatrix(test_features)
    fold_test = np.mean([m.predict(dtest) for m in fold_models], axis=0)
    test_preds += fold_test / kf.n_splits

# ==============================================================
# STEP 6: Evaluate OOF SMAPE
# ==============================================================
final_smape = smape(y, oof_preds)
print(f"\n🎯 Final OOF SMAPE: {final_smape:.4f}")

# ==============================================================
# STEP 7: Save submission
# ==============================================================
submission_file = "submission_xgb_gpu_final.csv"
submission = pd.DataFrame({
    'sample_id': test_img_df['sample_id'][:min_test],
    'price': test_preds
})
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved as {submission_file}")



🌟 Fold 1


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [21:23:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [21:23:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:29.72697	eval-rmse:32.88360
[100]	train-rmse:16.58931	eval-rmse:31.42808
[200]	train-rmse:11.16268	eval-rmse:31.08255
[300]	train-rmse:8.10108	eval-rmse:30.92045
[400]	train-rmse:6.02515	eval-rmse:30.85541
[500]	train-rmse:4.53676	eval-rmse:30.80877
[600]	train-rmse:3.43726	eval-rmse:30.77529
[700]	train-rmse:2.67103	eval-rmse:30.76295
[800]	train-rmse:2.05388	eval-rmse:30.75158
[900]	train-rmse:1.58251	eval-rmse:30.74510
[1000]	train-rmse:1.22959	eval-rmse:30.73969
[1100]	train-rmse:0.96043	eval-rmse:30.73659
[1200]	train-rmse:0.77188	eval-rmse:30.73341
[1300]	train-rmse:0.61497	eval-rmse:30.73170
[1400]	train-rmse:0.49172	eval-rmse:30.73086
[1500]	train-rmse:0.39734	eval-rmse:30.73037
[1600]	train-rmse:0.32075	eval-rmse:30.72980
[1700]	train-rmse:0.25912	eval-rmse:30.72954
[1800]	train-rmse:0.21223	eval-rmse:30.72891
[1900]	train-rmse:0.17306	eval-rmse:30.72849
[2000]	train-rmse:0.14407	eval-rmse:30.72833
[2100]	train-rmse:0.12252	eval-rmse:30.72831
[2200]	train-rmse:0

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\core.py:729: UserWarning: [21:32:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)


✅ Fold 1 SMAPE: 70.8625


...

[0]	train-rmse:30.91165	eval-rmse:32.89252
[100]	train-rmse:17.79568	eval-rmse:30.69112
[200]	train-rmse:12.02553	eval-rmse:30.18894
[300]	train-rmse:8.60537	eval-rmse:29.98490
[400]	train-rmse:6.38206	eval-rmse:29.89386
[500]	train-rmse:4.98018	eval-rmse:29.84234
[600]	train-rmse:3.75827	eval-rmse:29.81349
[700]	train-rmse:2.86070	eval-rmse:29.79339
[800]	train-rmse:2.21065	eval-rmse:29.78363
[900]	train-rmse:1.67898	eval-rmse:29.77753
[1000]	train-rmse:1.27767	eval-rmse:29.77103
[1100]	train-rmse:0.96286	eval-rmse:29.76635
[1200]	train-rmse:0.74015	eval-rmse:29.76537
[1300]	train-rmse:0.55725	eval-rmse:29.76447
[1400]	train-rmse:0.41581	eval-rmse:29.76289
[1500]	train-rmse:0.30679	eval-rmse:29.76188
[1600]	train-rmse:0.22811	eval-rmse:29.76108
[1700]	train-rmse:0.16618	eval-rmse:29.76061
[1800]	train-rmse:0.12065	eval-rmse:29.76022
[1900]	train-rmse:0.08730	eval-rmse:29.76007
[2000]	train-rmse:0.06306	eval-rmse:29.75986
[2100]	train-rmse:0.04633	eval-rmse:29.75984
[2200]	train-rmse:0

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\core.py:729: UserWarning: [21:58:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  return func(**kwargs)


✅ Fold 1 SMAPE: 70.4602


C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [21:58:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  self.starting_round = model.num_boosted_rounds()
C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\xgboost\callback.py:386: UserWarning: [21:58:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:38.53967	eval-rmse:32.90264
[100]	train-rmse:21.62710	eval-rmse:31.27232
[200]	train-rmse:14.56328	eval-rmse:30.78179
